# SpatioCube end-to-end demo (MouseBrain)

这个 notebook 演示：**读取合并 h5ad → 按 `sampleid` 拆片 → 相邻切片 OT 对齐（可选 middle-out 减轻漂移）→ 3D 聚类（Graph Diffusion + 图正则 GMM）→ 3D 可视化（离散色盲友好配色）**。

## 环境依赖（建议 conda 环境）

最小（能跑通本 notebook 主流程）：

- Python >= 3.9
- `scanpy`, `anndata`, `numpy`, `scipy`, `pandas`, `scikit-learn`
- OT 对齐：`POT`（可选但本 notebook 默认会用）
- 可视化：`plotly`

可选：`python-igraph` + `leidenalg`（仅当你要跑对照 Leiden）；对比学习 embedding 需要 `torch`。


## 0) 配置数据路径

把你的合并 `.h5ad` 路径填到下面（或设置环境变量 `SPATIOCUBE_MOUSEBRAIN_H5AD`）。


In [2]:
# 自动重载
%load_ext autoreload
%autoreload 2

In [3]:
import os

# 直接在这里指定
H5AD_PATH = os.environ.get(
    "SPATIOCUBE_MOUSEBRAIN_H5AD",
    "/cluster3/labData/jiamao/MouseBrain/h5ad/T286_T299_SCT_merge_bayes_anno.h5ad",
)
os.environ["SPATIOCUBE_MOUSEBRAIN_H5AD"] = H5AD_PATH
print("SPATIOCUBE_MOUSEBRAIN_H5AD =", os.environ["SPATIOCUBE_MOUSEBRAIN_H5AD"])

SPATIOCUBE_MOUSEBRAIN_H5AD = /cluster3/labData/jiamao/MouseBrain/h5ad/T286_T299_SCT_merge_bayes_anno.h5ad


## 1) 读取数据并拆片

- 默认按 `adata.obs['sampleid']` 拆片。
- 如果 `obsm['spatial']` 不存在，会从 `obs['coor_x_ad2'] / obs['coor_y_ad2']` 自动补。


In [6]:
import spatiocube as scb

adata = scb.read_merged_h5ad()  # uses env var

# ---（可选）SPARK-X：按 sampleid 分片筛选 SVG，再合并---
# 重要：如果多切片坐标是“每片各自的 2D 坐标”（常见），必须分片跑；
# 否则把所有切片堆在一起会扭曲空间距离（不同切片但坐标数值接近会被当成邻近点）。
# 不想用就把这一段注释掉。

# 每片取 top_n 个 SVG
sparkx_top_n = 2000
# 只保留至少出现在 min_presence 个切片中的基因（1=并集；更大=更稳更保守）
sparkx_min_presence = 2

sparkx_gene_presence = {}  # gene -> how many slices selected it
sample_ids = list(adata.obs["sampleid"].unique())
print("SPARK-X per-slice: n_slices =", len(sample_ids))

for sid in sample_ids:
    ad = adata[adata.obs["sampleid"] == sid].copy()
    print("  running SPARK-X on", sid, "n_obs=", ad.n_obs)

    res = scb.run_sparkx(
        ad,
        spatial_key="spatial",
        rscript="Rscript",
        num_cores=8,
        # 过滤加速（建议按数据规模调整）
        min_spot_total_counts=10,
        min_gene_nonzero_spots=20,
        min_gene_total_counts=50,
        max_genes=5000,
    )
    svg = res.top_genes(n=sparkx_top_n)
    for g in svg:
        sparkx_gene_presence[g] = sparkx_gene_presence.get(g, 0) + 1

sparkx_svg = sorted([g for g, c in sparkx_gene_presence.items() if c >= sparkx_min_presence])
print(
    "SPARK-X merged SVG genes:",
    len(sparkx_svg),
    "(min_presence=", sparkx_min_presence, ")",
    "Top10:",
    sparkx_svg[:10],
)
adata = adata[:, sparkx_svg].copy()
print("Filtered adata shape:", adata.shape)

cube = scb.SpatioCube.from_merged_h5ad(
    adata,
    slice_key="sampleid",
    lambda_z=0.01,
    # XY：后续 `align_adjacent_slices_ot` 会对每片做刚性旋转+平移，把不同坐标系叠到同一参考系
    # Z：按切片顺序分层；`z_spacing` 控制层间距（XY 很大时建议调大，否则 3D 观感会很“扁”）
    z_spacing=10.0,
    z_base=0.0,
    # 不信任 sampleid 的顺序：用表达相似自动推断切片前后关系
    order_mode="infer",
    # 推断顺序时使用 OT 距离 + 全局最短路径（强调全局流畅性）
    order_config=scb.OrderConfig(subsample_n=2000, svd_dim=50, knn=30, use_ot=True, ot_reg=0.05),
)

print("n_slices =", len(cube.adatas))
print("first 5 slice infos:")
for info in cube.slice_infos()[:5]:
    print(info)

SPARK-X per-slice: n_slices = 11
  running SPARK-X on T286 n_obs= 6919
  running SPARK-X on T287 n_obs= 7094
  running SPARK-X on T288 n_obs= 7174
  running SPARK-X on T291 n_obs= 8190
  running SPARK-X on T292 n_obs= 8231
  running SPARK-X on T293 n_obs= 8313
  running SPARK-X on T294 n_obs= 8224
  running SPARK-X on T295 n_obs= 8705
  running SPARK-X on T296 n_obs= 8921
  running SPARK-X on T297 n_obs= 8805
  running SPARK-X on T299 n_obs= 9566
SPARK-X merged SVG genes: 2930 (min_presence= 2 ) Top10: ['0610012G03Rik', '0610037L13Rik', '1110004F10Rik', '1110008F13Rik', '1110008P14Rik', '1110065P20Rik', '1500011B03Rik', '1500011K16Rik', '1700001L19Rik', '1700020I14Rik']
Filtered adata shape: (90142, 2930)


/cluster2/huanglab/jiamao/conda/envs/spatiocube/lib/python3.10/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/cluster2/huanglab/jiamao/conda/envs/spatiocube/lib/python3.10/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


n_slices = 11
first 5 slice infos:
SliceInfo(key='T299', z=0.0, n_obs=9566, extra=None)
SliceInfo(key='T297', z=10.0, n_obs=8805, extra=None)
SliceInfo(key='T296', z=20.0, n_obs=8921, extra=None)
SliceInfo(key='T295', z=30.0, n_obs=8705, extra=None)
SliceInfo(key='T294', z=40.0, n_obs=8224, extra=None)


In [7]:
# ---- 顺序诊断输出：推断的切片顺序 vs 你认知的真实顺序 ----
# 推断后 cube.adatas 的当前顺序（用于后续对齐）
inferred_order = [a.obs["sampleid"].iloc[0] for a in cube.adatas]
print("Inferred sampleid order:")
print(inferred_order)

# 如果你有真实顺序（例如 list[str]），可在这里粘贴对照：
# true_order = ["T300", "T301", ...]
# print("True order:")
# print(true_order)

# 打印相邻对的 spot 数，辅助判断是否存在明显不匹配的相邻对
print("Adjacent pairs (n_obs):")
for i in range(len(cube.adatas) - 1):
    a_tgt = cube.adatas[i]
    a_src = cube.adatas[i + 1]
    print(
        f"{i}: {a_src.obs['sampleid'].iloc[0]} -> {a_tgt.obs['sampleid'].iloc[0]}  "
        f"(src_n={a_src.n_obs}, tgt_n={a_tgt.n_obs})"
    )

Inferred sampleid order:
['T299', 'T297', 'T296', 'T295', 'T294', 'T293', 'T292', 'T291', 'T288', 'T287', 'T286']
Adjacent pairs (n_obs):
0: T297 -> T299  (src_n=8805, tgt_n=9566)
1: T296 -> T297  (src_n=8921, tgt_n=8805)
2: T295 -> T296  (src_n=8705, tgt_n=8921)
3: T294 -> T295  (src_n=8224, tgt_n=8705)
4: T293 -> T294  (src_n=8313, tgt_n=8224)
5: T292 -> T293  (src_n=8231, tgt_n=8313)
6: T291 -> T292  (src_n=8190, tgt_n=8231)
7: T288 -> T291  (src_n=7174, tgt_n=8190)
8: T287 -> T288  (src_n=7094, tgt_n=7174)
9: T286 -> T287  (src_n=6919, tgt_n=7094)


## 2) 相邻切片 OT 对齐（coarse-to-fine）

**切片顺序（全局）**：若 `order_mode="infer"`，会用各片表达的 pairwise 距离 + Held–Karp 求一条整体最顺的线性顺序（与 XY 无关）。

**对齐本身（仍是逐对刚性）**：每步只在 **一对切片** 上做表达 OT + 2D 刚性（旋转+平移），把源片坐标系叠到目标片上。
- `strategy="sequential"`：`1→0, 2→1, …` 链式叠到第 0 片；小误差会沿序列累积，长序列上易出现“螺旋/漂移”感。
- `strategy="middle_out"`（本 notebook 默认）：从中间锚片向两侧传播，通常能明显减轻累积漂移（仍非全局 bundle adjustment）。

为了适配 ~1e5 spots，内部会 **subsample** 做 OT，再把刚性变换应用到全量坐标。


In [8]:
from spatiocube.align import align_adjacent_slices_ot

# 关键：用表达 embedding KNN 选候选匹配，避免初始坐标偏差导致“对齐不动”
# middle_out：从中间切片向两侧链式对齐，减轻 sequential 链式累积误差导致的“螺旋漂移”
# shared_embedding=True：只 fit 一次全局 SVD，再 transform 每片（显著加速，默认开启）
# 方案 A（器官通用、无 2D 聚类标签）：对每片的表达 embedding 做 2D 空间图平滑，强化边界/分区结构信号
results = align_adjacent_slices_ot(
    cube,
    strategy="middle_out",
    shared_embedding=True,
    feature_mode="svd_smooth",
    smooth_k=15,
    smooth_alpha=0.7,
    smooth_steps=2,
    subsample_n=1200,
    svd_dim=50,
    expr_knn=30,
    ot_reg=0.05,
    ot_num_iter_max=5000,
    ot_stop_thr=1e-6,
    clip_quantile=0.95,
    n_iter=2,
    random_state=0,
)

for r in results[:5]:
    print(r)

print("has map_to_prev:", ["map_to_prev" in a.uns.get("SpatioCube", {}) for a in cube.adatas[:5]])

/cluster2/huanglab/jiamao/Project/SpatioCube/src/spatiocube/align.py:335: RuntimeWarning: Falling back to Sinkhorn OT for this adjacent pair: requested transport='emd' but subsample sizes (1200, 678) are incompatible with linear assignment or exceed `emd_max_n=2000`.
  warnings.warn(
/cluster2/huanglab/jiamao/conda/envs/spatiocube/lib/python3.10/site-packages/ot/unbalanced/_sinkhorn.py:1056: UserWarning: Numerical errors at iteration 0
  warnings.warn("Numerical errors at iteration %s" % cpt)
/cluster2/huanglab/jiamao/conda/envs/spatiocube/lib/python3.10/site-packages/ot/unbalanced/_sinkhorn.py:1075: UserWarning: Stabilized Unbalanced Sinkhorn did not converge.Try a larger entropy `reg` or a lower mass `reg_m`.Or a larger absorption threshold `tau`.
  warnings.warn(


AlignResult(chamfer_xy=9.516398471166031, n_source=8224, n_target=8313, method='coarse_emd_rigid_spatial_unbalanced')
AlignResult(chamfer_xy=4.9608976163102, n_source=8705, n_target=8224, method='coarse_emd_rigid_spatial_unbalanced')
AlignResult(chamfer_xy=5.140525822964749, n_source=8921, n_target=8705, method='coarse_emd_rigid_spatial_unbalanced')
AlignResult(chamfer_xy=5.798767659743025, n_source=8805, n_target=8921, method='coarse_emd_rigid_spatial_unbalanced')
AlignResult(chamfer_xy=33821.98229130601, n_source=9566, n_target=8805, method='coarse_sinkhorn_rigid_spatial_unbalanced')
has map_to_prev: [False, True, True, True, True]


In [9]:
# （推荐替代 BA）全局对齐：所有切片直接对齐到“锚切片”（避免链式累积导致螺旋）
# 这个方法通常比 BA 更稳健：不会出现“半边好、半边螺旋”的链式漂移。

from spatiocube.align import align_slices_to_anchor_ot

anchor_info = align_slices_to_anchor_ot(
    cube,
    # 默认锚为中间切片；你也可以手动指定，例如 0 或 len(cube.adatas)//2
    anchor_index=None,
    subsample_n=1200,
    svd_dim=50,
    expr_knn=30,
    ot_reg=0.1,
    clip_quantile=0.95,
    n_iter=2,
    random_state=0,
    shared_embedding=True,
    refresh_mapping=True,
)
print(anchor_info.get("status"), "anchor_index=", anchor_info.get("anchor_index"))

ok anchor_index= 5


In [10]:
import numpy as np

def bbox(xy):
    return xy[:,0].min(), xy[:,0].max(), xy[:,1].min(), xy[:,1].max()

for i, a in enumerate(cube.adatas[:5]):
    xy = np.asarray(a.obsm[cube.spatial_key])
    print(i, a.obs["sampleid"].iloc[0], "bbox:", bbox(xy))

0 T299 bbox: (673.5814729249415, 782.3907651094723, 130.74664208205462, 249.4094055281332)
1 T297 bbox: (672.7160635809114, 771.822338828381, 134.640091214524, 252.9757981372861)
2 T296 bbox: (674.6171491150883, 779.5559669214259, 130.60290742999484, 247.59429475015565)
3 T295 bbox: (674.4907439383347, 785.7905240073021, 133.50737199273829, 247.5945507713896)
4 T294 bbox: (677.7516080584372, 781.3604373308816, 131.08232780572914, 243.56768059416072)


## 3) 3D 聚类（推荐：Graph Diffusion + Graph-regularized GMM）

相比 Leiden（纯图划分），这个方法会：

- 先在融合图上做 **跨切片扩散**（把相邻切片信息沿 mapping/inter 边传播，提升稳健性）
- 再做 **图正则 GMM（mean-field EM）**，让空间近邻/跨片对应更倾向同簇

关键超参直觉：

- `beta_map`：跨片 mapping 边的权重（越大越强调跨片一致性）
- `eta`：扩散强度（越大信息传播越强，过大可能过平滑）
- `lam`：聚类阶段的图一致性强度（越大结果更平滑）


In [11]:
# --- 推荐：Graph Diffusion + Graph-regularized GMM ---
# 需要对齐步骤已经写入 map_to_prev（上一节 align_adjacent_slices_ot 会做）

import spatiocube as scb

res = scb.cluster_3d_diffusion_gmm(
    cube,
    n_clusters=20,
    svd_dim=50,
    # 图融合权重：intra / mapping / inter
    alpha_intra=1.0,
    beta_map=2.0,
    gamma_inter=1.0,
    # 扩散强度
    eta=0.5,
    diffusion_steps=50,
    diffusion_tol=1e-6,
    # 图正则聚类强度
    lam=1.0,
    em_iters=30,
    init="kmeans",
    random_state=0,
    write_back=True,
)

labels = res.labels
print("n_labels =", len(set(labels)))
print("first slice cluster counts:")
print(cube.adatas[0].obs[cube.cluster_key].value_counts().head())

# --- baseline：Leiden（保留作对照）---
# from spatiocube.graph import build_3d_adjacency, leiden_cluster
# A = build_3d_adjacency(cube, n_intra=15, n_inter=5, prefer_mapping=True)
# labels = leiden_cluster(A, resolution=1.0, random_state=0)
# cube.set_clusters(labels)


n_labels = 20
first slice cluster counts:
SpatioCube_cluster
1     2571
12    1253
7     1189
3      985
16     792
Name: count, dtype: int64


## 4) 3D 可视化（Plotly）

把所有切片的 `spatial_3d` 拼起来画点云，颜色按 `SpatioCube_cluster`。**聚类为离散标签**：使用 Okabe–Ito 固定配色 + 每类单独 trace（图例即类别 id），避免 Viridis 渐变色。


In [12]:
import numpy as np

xy0 = np.asarray(cube.adatas[0].obsm[cube.spatial_key])
xy1 = np.asarray(cube.adatas[1].obsm[cube.spatial_key])
print("slice0 bbox:", xy0.min(0), xy0.max(0))
print("slice1 bbox:", xy1.min(0), xy1.max(0))

slice0 bbox: [673.58147292 130.74664208] [782.39076511 249.40940553]
slice1 bbox: [672.71606358 134.64009121] [771.82233883 252.97579814]


In [14]:
import numpy as np

cube.write_back()  # ensure spatial_3d
xyz = np.vstack([a.obsm[cube.spatial_3d_key] for a in cube.adatas])
c = np.concatenate([a.obs[cube.cluster_key].to_numpy() for a in cube.adatas])

# discrete_colors=True：每类固定 Okabe–Ito 色 + 分 trace 图例（非 Viridis 渐变色）
fig = scb.plotly_pointcloud(
    xyz, color=c, size=2.0, title="SpatioCube 3D clusters", discrete_colors=True
)

# 远程/无前端的环境下，若 `fig.show()` 的 mime 渲染不稳定，可改为导出 html 再用浏览器打开：
fig.write_html("spatiocube_3d_clusters.html")
fig.show()

## 5)（可选）对比学习 embedding（需要 torch）

如果你安装了 `torch`，可以在 3D 图上先学一个对比学习 embedding `X_spatiocube`。
然后在 `X_spatiocube` 上再跑 Leiden/kmeans（此处示例只展示如何生成 embedding）。


In [ ]:
# from spatiocube.contrastive import contrastive_embed_3d, ContrastiveConfig
# Z = contrastive_embed_3d(cube, A, config=ContrastiveConfig(epochs=20, batch_size=4096))
# print("Z shape =", Z.shape)
# print("per-slice obsm key:", cube.adatas[0].obsm["X_spatiocube"].shape)
pass